# llm from scratch

### imports

In [81]:
import re
import tiktoken

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### Raw text

In [38]:
with open('data/the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('total number of characters:', len(raw_text))

total number of characters: 20479


In [39]:
print(raw_text[:1000])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through th

### Preprocessing

In [40]:
preprocessed_text = re.split(r'([,.?_!"()\']|--|\s)', raw_text)

In [41]:
print(preprocessed_text[:100])

['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--', 'so', ' ', 'it', ' ', 'was', ' ', 'no', ' ', 'great', ' ', 'surprise', ' ', 'to', ' ', 'me', ' ', 'to', ' ', 'hear', ' ', 'that', ',', '', ' ', 'in', ' ', 'the', ' ', 'height', ' ', 'of', ' ', 'his', ' ', 'glory', ',', '', ' ', 'he', ' ', 'had', ' ', 'dropped', ' ', 'his', ' ', 'painting', ',', '', ' ', 'married', ' ', 'a', ' ', 'rich', ' ', 'widow', ',', '', ' ', 'and', ' ', 'established', ' ', 'himself', ' ', 'in', ' ', 'a', ' ']


Remove whitespaces

In [42]:
preprocessed_text = [item.strip() for item in preprocessed_text if item.strip() != '']
print(len(preprocessed_text))   

4649


In [43]:
preprocessed_text[:100]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in',
 'the',
 'height',
 'of',
 'his',
 'glory',
 ',',
 'he',
 'had',
 'dropped',
 'his',
 'painting',
 ',',
 'married',
 'a',
 'rich',
 'widow',
 ',',
 'and',
 'established',
 'himself',
 'in',
 'a',
 'villa',
 'on',
 'the',
 'Riviera',
 '.',
 '(',
 'Though',
 'I',
 'rather',
 'thought',
 'it',
 'would',
 'have',
 'been',
 'Rome',
 'or',
 'Florence',
 '.',
 ')',
 '"',
 'The',
 'height',
 'of',
 'his',
 'glory',
 '"',
 '--',
 'that',
 'was',
 'what',
 'the',
 'women',
 'called',
 'it',
 '.',
 'I',
 'can',
 'hear',
 'Mrs',
 '.',
 'Gideon',
 'Thwing',
 '--',
 'his',
 'last',
 'Chicago',
 'sitter',
 '--']

### Tokenization with simple tokenizer

In [44]:
all_tokens = sorted(list(set(preprocessed_text)))
print(len(all_tokens))

1159


In [45]:
print(all_tokens[:100])

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Carlo;', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed', 'Devonshire', 'Don', 'Dubarry', 'Emperors', 'Florence', 'For', 'Gallery', 'Gideon', 'Gisburn', 'Gisburns', 'Grafton', 'Greek', 'Grindle', 'Grindle:', 'Grindles', 'HAD', 'Had', 'Hang', 'Has', 'He', 'Her', 'Hermia', 'His', 'How', 'I', 'If', 'In', 'It', 'Jack', 'Jove', 'Just', 'Lord', 'Made', 'Miss', 'Money', 'Monte', 'Moon-dancers', 'Mr', 'Mrs', 'My', 'Never', 'No', 'Now', 'Nutley', 'Of', 'Oh', 'On', 'Once', 'Only', 'Or', 'Perhaps', 'Poor', 'Professional', 'Renaissance', 'Rickham', 'Rickham;', 'Riviera', 'Rome', 'Russian', 'Sevres', 'She', 'Stroud', 'Strouds', 'Suddenly', 'That', 'The', 'Then', 'There', 'There:']


In [46]:
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

In [47]:
print(len(all_tokens))

1161


In [48]:
token2idx = {token: idx for idx, token in enumerate(all_tokens)}    

In [49]:
for idx, item in enumerate(list(token2idx.items())[-10:]):
    print(item)

('year', 1151)
('years', 1152)
('yellow', 1153)
('yet', 1154)
('you', 1155)
('younger', 1156)
('your', 1157)
('yourself', 1158)
('<|endoftext|>', 1159)
('<|unk|>', 1160)


In [50]:
class SimpleTokenizer:
    def __init__(self, token2idx):
        self.token2idx = token2idx
        self.idx2token = {idx: token for token, idx in token2idx.items()}
    
    def tokenize(self, text):
        preprocessed_text = re.split(r'([,.?_!"()\']|--|\s)', text) 
        preprocessed_text = [item.strip() for item in preprocessed_text if item.strip() != '']
        return [self.token2idx.get(token, self.token2idx['<|unk|>']) for token in preprocessed_text]
    
    def detokenize(self, idxs):
        text = " ".join([self.idx2token[idx] for idx in idxs])
        text = re.sub(r'\s+([,.?_!"()\']|--|\s)', r'\1', text)
        return text

In [51]:
simple_tokenizer = SimpleTokenizer(token2idx)

In [52]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [53]:
print(simple_tokenizer.tokenize(text))

[1160, 5, 362, 1155, 642, 1000, 10, 1159, 57, 1013, 981, 1009, 738, 1013, 1160, 7]


In [54]:
print(simple_tokenizer.detokenize(simple_tokenizer.tokenize(text))) 

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### BPE with tiktoken tokenizer

In [58]:
tokenizer = tiktoken.get_encoding("gpt2")

In [63]:
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [66]:
idxs = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
idxs

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 262,
 20562,
 13]

In [67]:
reconstructed_text = tokenizer.decode(idxs) 
reconstructed_text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

### Dataset and dataloader

In [74]:
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.tokenizer = tokenizer
        self.token_ids = tokenizer.encode(text)
        self.input_ids = []
        self.target_ids = []

        for i in range(0, len(self.token_ids) - max_length, stride):
            input_chunk = self.token_ids[i:i+max_length]
            target_chunk = self.token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [75]:
def create_dataloader(text, batch_size=4, max_length=256,
                        stride=128, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, max_length, stride)   
    dataloader = DataLoader(dataset, batch_size=batch_size, 
                shuffle=shuffle, drop_last=drop_last)
    return dataloader

In [78]:
with open('data/the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

dataloader = create_dataloader(raw_text, batch_size=4, max_length=256,
                                stride=1, shuffle=True, drop_last=True)     

In [80]:
trial_batch = next(iter(dataloader))
trial_batch

[tensor([[10032,   286,   852,  ...,   832,  6164,    25],
         [  286,   326, 17548,  ...,   655,  9617,  7521],
         [ 3465,   644,  1245,  ...,   683,    25,   262],
         [  606,   286,   511,  ...,    13,   198,   198]]),
 tensor([[  286,   852, 13055,  ...,  6164,    25,   366],
         [  326, 17548,   286,  ...,  9617,  7521,   656],
         [  644,  1245,   262,  ...,    25,   262,  1109],
         [  286,   511,  6799,  ...,   198,   198,     1]])]

### Token embeddings

In [ ]:
vocab_size = 50257  
embedding_dim = 256
embedding_lager = nn.Embedding(vocab_size, embedding_dim)

### Approx GELU

In [82]:
class ApproxGELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(0.79788456 * (x + 0.044715 * x**3)))   

### Model configuration

GPT_CONFIG